# Paired evaluation of review controls

CPU evaluation; uses the existing versioned ECG assertion rules and machine references.
Report coverage before complete-case micro-F1, omissions, reference-discordant positives, and copy baselines.
Intervals are exploratory patient-cluster bootstrap intervals, not adjusted for multiplicity.
Only completed new-run results are compared; no old successes are mixed into the new protocol.
Measured processing/generation latency and allocated memory do not establish energy savings or workflow cost.
No demographic-fairness or clinician harm claim follows from these experiments.
Use partial evaluation only for pilots. Final submission needs complete planned-attempt coverage, even if some attempts fail.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get("EXPERIMENT4_CODE_ROOT", str(Path.cwd())))
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    CODE_ROOT = Path.cwd()
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    raise FileNotFoundError("Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 code folder")
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
os.environ.setdefault("EXPERIMENT4_ROOT", str(Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", str(CODE_ROOT.parent / "artifacts"))) / "experiment4"))
os.environ.setdefault("HF_HOME", str(CODE_ROOT.parent / ".cache" / "huggingface"))
from e4_review_controls import roots
SOURCE_ROOT, REVIEW_ROOT = roots()
print({"source_read_only": str(SOURCE_ROOT), "new_output": str(REVIEW_ROOT)})


In [ ]:
# Evaluate PULSE v2 and the unchanged original Qwen run.
# No fallback to the original PULSE pilot if the new folder is missing.
EVALUATION_ROOT = REVIEW_ROOT / "pulse_prompt_v2"
if not (EVALUATION_ROOT / "prompt_revision.json").is_file():
    raise FileNotFoundError("Run updated Notebook 20 preparation first")
from e4_review_controls import evaluate
status = evaluate(dest=EVALUATION_ROOT,
                  bootstraps=int(os.environ.get("E4_REVIEW_BOOTSTRAPS", "1000")),
                  allow_partial=os.environ.get("E4_REVIEW_ALLOW_PARTIAL", "0") == "1")
status


In [ ]:
import pandas as pd
for name in ["coverage.csv", "summary.csv", "paired_comparisons.csv"]:
    path = EVALUATION_ROOT / "evaluation" / name
    if path.exists():
        print(name)
        display(pd.read_csv(path))


## Interpretation gate
1. A trimodal-versus-raw ECG result changes retrieval **and** representation learning. The bridge ECG-only contrast isolates scoring within the learned space.
2. Full-versus-same isolates added text after the same retrieval. It does not establish a benefit of multimodal training.
3. Linked-versus-broken tests sensitivity to experimentally corrupted coherence while preserving ECG anchors and the global CXR/lab multiset. It does not establish causal clinical relationships.
4. Compare generation with nearest-neighbor copying on common cases. Strong copy performance indicates transfer of diagnostic vocabulary may explain part of a gain.
5. Compare PULSE rendering conditions on identical cases, disclose failures, and avoid generalizing to other specialist interfaces.
6. Interpret reproduced results alongside completion, paired-case coverage, and protocol provenance before updating manuscript tables.